# Test: Additional Properties Tab vs. `view-details` -- FINAL RECORD (v11)

> **STATUS: QUESTION ANSWERED (2026-07-27, on the temporary `datacatalog-b` server).**
> `view-details` DOES include Additional-Properties-Tab property values, but NOT in
> `summaryPropertyMap` -- they live in `propertyInfo.customTabPropertyMap`. See the
> **Conclusions** section at the bottom for full findings and caveats.


**Question:** Are there property groups configured to show in the *Additional
Properties Tab* (rather than the Summary tab), and if so, does `view-details`
return their property values anyway, or are they missing?

**Why this matters:** Step 0's probe pulled property values from
`view-details`'s own `propertyInfo.summaryPropertyMap` for many groups
(`ODS DB Link`, `Access Role Request`, `Data Classification`, `Business Unit`,
`Update Frequency`, POC contacts). That's strong evidence `view-details`
carries a wide range of properties -- but every one of those groups *could*
happen to be configured for the Summary tab. This notebook looks specifically
for a group that is **not**, and checks whether `view-details` includes it
anyway.

**Endpoints used** (confirmed from the `property-management` section of
Swagger UI):

- `GET /public/api/property-management/groups` -- all property groups
- `GET /public/api/property-management/groups/{groupId}` -- group details
  (this is where the Summary vs. Additional Properties Tab config should live
  -- exact field name TBD, inspected in Step 2 below)
- `GET /public/api/property-management/groups/{groupId}/elements/{elementType}`
  -- paginated property values for the elements (views) assigned to a group
- `GET /public/api/view-details` -- from Step 0, keyed on
  `(viewName, databaseName, serverId)`


## Step 0 -- Auth setup (same as your Step 0 probe notebook)

There is no separate `denodo_client.py` / `oauth_manager.py` module -- `OAuthManager`
is defined inline, and calls go straight through `auth_manager.get(full_url)` with no
wrapper class. This cell is copy-pasted from your existing setup so this notebook can
run standalone.

**TEMPORARY SERVER CONFIG (updated 2026-07-27):** `datacatalog-d.lanl.gov` is still down (alias not yet moved to the new server — confirmed by Maxen on 2026-07-23). Diagnostic on 2026-07-27 showed that `den-datacat-d.lanl.gov` is up but only accepts Basic Auth (`www-authenticate: Basic realm="Denodo"` → 401 for Bearer tokens), while `datacatalog-b.lanl.gov` accepts the OAuth Bearer token and returns data (200). This notebook therefore uses **`datacatalog-b.lanl.gov`** as BOTH the OAuth redirect host and the API root. Both lines are marked TEMPORARY in the cell below — revert them once the `datacatalog-d` alias is restored.


In [1]:
import html
import json
import os
import time

import requests
import truststore
import webview
from requests_oauthlib import OAuth2Session

truststore.inject_into_ssl()  # handles LANL internal TLS certs

required = ["AUTH_FLOW_CLIENT_ID", "AUTH_FLOW_CLIENT_SECRET",
            "REDIRECT_URI", "AUTH_URL", "TOKEN_URL", "SCOPE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")
print("All 6 env vars present \u2713")


class OAuthManager:
    """Complete OAuth manager that handles initial auth AND refresh"""

    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None

    def authenticate(self):
        self.oauth = OAuth2Session(self.client_id, redirect_uri=self.redirect_uri, scope=self.scope)
        authorization_url, state = self.oauth.authorization_url(self.auth_url)
        print("Authenticating...")
        print(f"[DEBUG] authorization_url = {authorization_url}")
        authorization_response = self._launch_browser_auth(authorization_url)
        if not authorization_response:
            print("\u2717 Authentication failed")
            return False
        self.token = self.oauth.fetch_token(
            self.token_url, authorization_response=authorization_response,
            client_secret=self.client_secret,
        )
        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)
        print("\u2713 Authentication successful")
        return True

    def _launch_browser_auth(self, authorization_url):
        authorization_response = None

        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            print(f"[DEBUG] page loaded: {current_url}")
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print("\u2713 Captured Authorization")
                window.hide()
                time.sleep(2.5)
                window.destroy()

        window = webview.create_window(
            "OAuth Authorization", authorization_url, width=800, height=600,
            resizable=True, on_top=True,
        )
        window.events.loaded += on_loaded
        webview.start(private_mode=True)  # never reuse a cached LANL SSO session
        return authorization_response

    def _is_token_expired(self):
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)

    def _refresh_access_token(self):
        if not self.refresh_token:
            print("\u26a0 No refresh token available, re-authenticating...")
            return self.authenticate()
        try:
            new_token = self.oauth.refresh_token(
                self.token_url, refresh_token=self.refresh_token,
                client_id=self.client_id, client_secret=self.client_secret,
            )
            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)
            print("\u2713 Token refreshed successfully")
            return True
        except Exception as e:
            print(f"\u2717 Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()

    def _ensure_authenticated(self):
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()

    def get(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.get(url, headers=headers, **kwargs)

    def post(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.post(url, headers=headers, **kwargs)


# TEMPORARY (2026-07-27): datacatalog-d.lanl.gov alias is still down.
# Diagnostic (2026-07-27) confirmed:
#   - den-datacat-d.lanl.gov  -> app up, but Bearer tokens rejected (401, www-authenticate: Basic)
#   - datacatalog-b.lanl.gov  -> accepts the OAuth Bearer token (200) -- correct API root for now
BASE_URL = "https://datacatalog-b.lanl.gov/denodo-data-catalog"  # TEMPORARY -- matches the redirect host
# BASE_URL = "https://datacatalog-d.lanl.gov/denodo-data-catalog"  # original dev -- restore when alias is fixed
TARGET_DB = "dataportal"
SERVER_ID = 1

# CONFIRMED via Swagger UI on datacatalog-b (2026-07-27):
# /groups/{groupId}/elements/{elementType} takes elementType = "VIEWS"
# (uppercase plural, per the Swagger dropdown) and REQUIRES limit + offset
# query params (integers) -- omitting them is what caused the earlier 400s.
ELEMENT_TYPE = "VIEWS"

auth_manager = OAuthManager(
    client_id=os.getenv("AUTH_FLOW_CLIENT_ID"),
    client_secret=os.getenv("AUTH_FLOW_CLIENT_SECRET"),
    # TEMPORARY (Maxen, 2026-07-23): override REDIRECT_URI env var while alias is down
    redirect_uri="https://datacatalog-b.lanl.gov/oauth/2.0/redirectURL.jsp",
    # redirect_uri=os.getenv("REDIRECT_URI"),  # restore when alias is fixed
    auth_url=os.getenv("AUTH_URL"),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv("SCOPE"),
)
auth_manager.authenticate()


All 6 env vars present ✓
Authenticating...
[DEBUG] authorization_url = https://idp.lanl.gov/as/authorization.oauth2?response_type=code&client_id=376bca25-336f-48d1-ae1d-fd68fa153196&redirect_uri=https%3A%2F%2Fdatacatalog-b.lanl.gov%2Foauth%2F2.0%2FredirectURL.jsp&scope=den-datacat-adm&state=QFMsbE4nZZ75zcFxlXywxAyl0X2rpj
[DEBUG] page loaded: https://weblogin.lanl.gov/login
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://datacatalog-b.lanl.gov/oauth/2.0/redirectURL.jsp?code=Z__w9CVoNSAl3XTSUXLZSdt7P7iZEq26yboAAAAC&state=QFMsbE4nZZ75zcFxlXywxAyl0X2rpj
✓ Captured Authorization
[DEBUG] page loaded: https://datacatalog-b.lanl.gov/oauth/2.0/redirectURL.jsp?code=Z__w9CVoNSAl3XTSUXLZSdt7P7iZEq26yboAAAAC&state=QFMsbE4nZZ75zcFxlXywxAyl0X2rpj
✓ Captured Authorization
✓ Authentication successful


True

In [2]:
# Sanity check (was the 401 diagnostic in v5 -- resolved 2026-07-27).
# Root cause: den-datacat-d only accepts Basic Auth; datacatalog-b accepts Bearer.
# This cell now just confirms the token works against the new BASE_URL before Step 1.
r = auth_manager.get(
    BASE_URL + "/public/api/view-details",
    params={"viewName": "announcement", "databaseName": "dataportal", "serverId": SERVER_ID},
)
print("view-details:", r.status_code)

r2 = auth_manager.get(BASE_URL + "/public/api/views", params={"serverId": SERVER_ID})
print("list views:  ", r2.status_code)

assert r.status_code == 200 and r2.status_code == 200, (
    "Token not accepted by BASE_URL -- do not continue to Step 1. "
    "Re-run the host diagnostic (v5) and check with Maxen."
)
print("Auth sanity check passed -- OK to continue to Step 1.")


view-details: 200
list views:   200
Auth sanity check passed -- OK to continue to Step 1.


## Step 1 -- List all property groups

`GET /public/api/property-management/groups`


In [3]:
resp = auth_manager.get(BASE_URL + "/public/api/property-management/groups")
resp.raise_for_status()
groups = resp.json()

print(f"Found {len(groups)} property group(s).")
for g in groups:
    print(f"  id={g.get('id')!s:>4}  name={g.get('name')}")


Found 9 property group(s).
  id=   1  name=Default Group
  id=  81  name=Additional Information
  id=  82  name=Details
  id= 101  name=API Information
  id= 121  name=Duplicate Table
  id= 141  name=View Details
  id= 161  name=ODS Details
  id= 162  name=Data Card
  id= 163  name=AI Portal


## Step 2 -- Inspect one group's details to find the display-location field

`GET /public/api/property-management/groups/{groupId}`

**RESOLVED (2026-07-27 run):** the display-location field is **`placeToShow`**,
observed value `"SUMMARY_TAB"` (e.g. Default Group). The constants in the next
cell are set accordingly. The cell below still prints one raw group-detail JSON
as a sanity check -- if the schema ever changes, re-derive the field from it.


In [4]:
if groups:
    sample_group_id = groups[0]["id"]
    resp = auth_manager.get(BASE_URL + f"/public/api/property-management/groups/{sample_group_id}")
    resp.raise_for_status()
    sample_detail = resp.json()
    print(json.dumps(sample_detail, indent=2))
else:
    print("No groups returned -- check auth/permissions before continuing.")


{
  "id": 1,
  "name": "Default Group",
  "description": "Default group to create properties.",
  "descriptionType": "TEXT",
  "placeToShow": "SUMMARY_TAB",
  "isDefault": true,
  "propertyCount": null
}


In [5]:
# CONFIRMED from the 2026-07-27 run (group-details JSON, e.g. Default Group):
#   "placeToShow": "SUMMARY_TAB"
# Any other value of placeToShow means the group displays somewhere other than
# the Summary tab (exact enum for the Additional Properties Tab still unknown --
# Step 3 will print whatever non-SUMMARY_TAB values exist in this catalog).
DISPLAY_LOCATION_FIELD = "placeToShow"
SUMMARY_VALUE = "SUMMARY_TAB"

def get_group_details(group_id):
    resp = auth_manager.get(BASE_URL + f"/public/api/property-management/groups/{group_id}")
    resp.raise_for_status()
    return resp.json()

def is_non_summary(group_detail):
    value = group_detail.get(DISPLAY_LOCATION_FIELD)
    if value is None:
        return None  # unknown -- field not found, needs a different key
    return value != SUMMARY_VALUE


## Step 3 -- Fetch details for every group and filter for non-Summary ones


In [6]:
group_details = []
for g in groups:
    detail = get_group_details(g["id"])
    group_details.append(detail)

unknown = [d for d in group_details if is_non_summary(d) is None]
non_summary_groups = [d for d in group_details if is_non_summary(d) is True]
summary_groups = [d for d in group_details if is_non_summary(d) is False]

# Show every group's raw placeToShow value first -- this reveals the actual
# enum values used in this catalog (SUMMARY_TAB vs. whatever else exists).
print("placeToShow per group:")
for d in group_details:
    print(f"  {d.get('name'):<30} -> {d.get(DISPLAY_LOCATION_FIELD)!r}")
print()

print(f"Summary-tab groups:     {len(summary_groups)}")
print(f"Additional-tab groups:  {len(non_summary_groups)}")
print(f"Unknown (check field):  {len(unknown)}")

if unknown:
    print("\nDISPLAY_LOCATION_FIELD did not match -- inspect this raw record:")
    print(json.dumps(unknown[0], indent=2))

for d in non_summary_groups:
    print(f"  - {d.get('name')} (id={d.get('id')})")


placeToShow per group:
  Default Group                  -> 'SUMMARY_TAB'
  Additional Information         -> 'SPECIFIC_CUSTOM_PROPERTY_TAB'
  Details                        -> 'SUMMARY_TAB'
  API Information                -> 'SUMMARY_TAB'
  Duplicate Table                -> 'SUMMARY_TAB'
  View Details                   -> 'SUMMARY_TAB'
  ODS Details                    -> 'SUMMARY_TAB'
  Data Card                      -> 'SPECIFIC_CUSTOM_PROPERTY_TAB'
  AI Portal                      -> 'SUMMARY_TAB'

Summary-tab groups:     7
Additional-tab groups:  2
Unknown (check field):  0
  - Additional Information (id=81)
  - Data Card (id=162)


## Step 4 -- For each non-Summary group, find an assigned view and its property values

`GET /public/api/property-management/groups/{groupId}/elements/{elementType}`

**CONFIRMED from Swagger UI on datacatalog-b (2026-07-27):**

- `elementType` (path, required): `"VIEWS"` (uppercase plural, per the Swagger dropdown)
- `limit` (query, **required**, int) and `offset` (query, **required**, int) --
  these being required is why the earlier calls returned 400 with an empty body
- `serverId` (query, optional, default 1); `databaseName` / `elementName` (query, optional filters)
- **Response shape:** a flat JSON array -- one row per *(element, property)* pair:
  `databaseId, databaseName, elementId, name, possibleValues, propertyId,
  propertyName, propertyType, searchValue, visualValue`. No pagination wrapper,
  so rows are regrouped per view below. Note `elementId` is returned here --
  recovering the id that this server's views list returns as null.


In [7]:
def get_group_elements(group_id, element_type=ELEMENT_TYPE, limit=200, offset=0):
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}/elements/{element_type}",
        params={"serverId": SERVER_ID, "limit": limit, "offset": offset},
    )
    if not resp.ok:
        print(f"[{resp.status_code}] {resp.url}")
        print("body:", resp.text[:500])
    resp.raise_for_status()
    return resp.json()  # flat list: one row per (element, property) pair


group_samples = {}  # group_id -> {"group_name", "view", "db", "element_id", "expected_properties"}

for d in non_summary_groups:
    group_id = d["id"]
    rows = get_group_elements(group_id)
    print(f"--- group '{d.get('name')}' (id={group_id}): {len(rows)} (element, property) row(s) ---")

    # RAW DUMP (v9): the v8 run showed propertyName == None on every row, so we
    # need to see the actual row schema instead of assuming Swagger's example.
    print(json.dumps(rows, indent=2)[:3000])

    if not rows:
        print(f"  no views have group '{d.get('name')}' populated -- skipping")
        continue

    # Regroup rows by element (view). Keep None-named properties visible but flagged.
    by_view = {}
    for row in rows:
        key = (row.get("name"), row.get("databaseName"))
        by_view.setdefault(key, {"element_id": row.get("elementId"), "properties": {}})
        by_view[key]["properties"][row.get("propertyName")] = row.get("visualValue")

    print(f"  distinct views in this page: {len(by_view)}")
    for (vname, vdb), info in list(by_view.items())[:5]:
        prop_names = list(info["properties"].keys())
        flag = "  <-- ALL property names are None (row schema mismatch?)" if prop_names == [None] else ""
        print(f"    {vdb}.{vname} (elementId={info['element_id']}): {prop_names}{flag}")

    (sample_name, sample_db), sample_info = next(iter(by_view.items()))
    group_samples[group_id] = {
        "group_name": d.get("name"),
        "view": sample_name,
        "db": sample_db or TARGET_DB,
        "element_id": sample_info["element_id"],
        "expected_properties": sample_info["properties"],  # {propertyName: visualValue}
    }

print(f"\nCollected {len(group_samples)} sample view(s) to check against view-details.")


--- group 'Additional Information' (id=81): 1 (element, property) row(s) ---
[
  {
    "propertyId": null,
    "propertyType": null,
    "propertyName": null,
    "databaseId": 5,
    "databaseName": "dataportal",
    "elementId": 5755,
    "name": "admin_option_type_fvts",
    "visualValue": null,
    "searchValue": null,
    "possibleValues": null
  }
]
  distinct views in this page: 1
    dataportal.admin_option_type_fvts (elementId=5755): [None]  <-- ALL property names are None (row schema mismatch?)
--- group 'Data Card' (id=162): 1 (element, property) row(s) ---
[
  {
    "propertyId": null,
    "propertyType": null,
    "propertyName": null,
    "databaseId": 5,
    "databaseName": "dataportal",
    "elementId": 414056,
    "name": "bv_oc_footprints_alldesc_esr",
    "visualValue": null,
    "searchValue": null,
    "possibleValues": null
  }
]
  distinct views in this page: 1
    dataportal.bv_oc_footprints_alldesc_esr (elementId=414056): [None]  <-- ALL property names are None

## Step 4b -- Resolve property names and get real expected values

**What the v9 raw dumps showed (2026-07-27):**

- The collection endpoint (Step 4) returns one row per element with ALL property
  fields null -- it identifies *which views* have the group assigned, nothing more.
- The per-element endpoint returns rows with **`propertyId` + `visualValue` only**
  (no `propertyName` field on this server -- different schema than assumed).
- Group 81 / `admin_option_type_fvts`: **propertyId 159 IS populated** (an HTML
  link to collaborate.lanl.gov). Ids 160, 161 are null.
- Group 162 / `bv_oc_footprints_alldesc_esr`: ids 282, 283 both null --
  Data Card genuinely unpopulated (consistent with the June finding).

So this step now does the missing join: fetch the group's property *definitions*
(`GET /groups/{groupId}/properties` -- verified working in the July 16 session,
returns `id` + `name` per property), map `propertyId -> name`, and build the
expected set from the populated values only.


In [8]:
def get_group_property_defs(group_id):
    """Property definitions (id, name, type, ...) for one group."""
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}/properties",
        params={"serverId": SERVER_ID},
    )
    if not resp.ok:
        print(f"[{resp.status_code}] {resp.url}")
        print("body:", resp.text[:500])
    resp.raise_for_status()
    return resp.json()


def get_element_properties(group_id, element_id):
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}"
                   f"/elements/VIEWS/{element_id}/properties",
        params={"serverId": SERVER_ID},
    )
    if not resp.ok:
        print(f"[{resp.status_code}] {resp.url}")
        print("body:", resp.text[:500])
    resp.raise_for_status()
    return resp.json()


for group_id, sample in group_samples.items():
    print(f"--- group '{sample['group_name']}' (id={group_id}) / "
          f"view {sample['db']}.{sample['view']} (elementId={sample['element_id']}) ---")

    defs = get_group_property_defs(group_id)
    id_to_name = {p.get("id"): p.get("name") for p in defs}
    print(f"  property definitions in group: "
          f"{ {p.get('id'): p.get('name') for p in defs} }")

    props = get_element_properties(group_id, sample["element_id"])

    all_assigned = {}   # propertyName -> visualValue (nulls included)
    for p in props:
        pname = id_to_name.get(p.get("propertyId"), f"propertyId_{p.get('propertyId')}")
        all_assigned[pname] = p.get("visualValue")

    populated = {k: v for k, v in all_assigned.items() if v not in (None, "")}

    sample["all_assigned_properties"] = all_assigned
    sample["expected_properties"] = populated  # only populated values are testable

    print(f"  all assigned properties: {list(all_assigned.keys())}")
    if populated:
        for k, v in populated.items():
            print(f"  POPULATED -> {k!r}: {v!r}")
    else:
        print("  no populated values on this element -- untestable sample for the gap question")
    print()


--- group 'Additional Information' (id=81) / view dataportal.admin_option_type_fvts (elementId=5755) ---
  property definitions in group: {159: 'More Help for Web Services', 160: 'Database Role', 161: 'Property_Implementation'}
  all assigned properties: ['More Help for Web Services', 'Database Role', 'Property_Implementation']
  POPULATED -> 'More Help for Web Services': '<p><a href="https://collaborate.lanl.gov/x/tYV4Cw">https://collaborate.lanl.gov/x/tYV4Cw</a></p>'

--- group 'Data Card' (id=162) / view dataportal.bv_oc_footprints_alldesc_esr (elementId=414056) ---
  property definitions in group: {282: 'Motivation', 283: 'Composition'}
  all assigned properties: ['Motivation', 'Composition']
  no populated values on this element -- untestable sample for the gap question



## Step 5 -- Call `view-details` for each sample view and search for the values

For every populated Additional-tab property, three checks against the FULL
`view-details` response (not just `propertyInfo.summaryPropertyMap`):

1. Is the property name a key inside `summaryPropertyMap`?
2. Does the property name appear anywhere in the serialized response?
3. Does the property **value** appear anywhere in the serialized response?
   (checked both raw and HTML-stripped -- the group-81 value is an HTML link,
   and `collaborate.lanl.gov/x/tYV4Cw` is distinctive enough that a value hit
   or miss is strong evidence either way)

A property whose value appears nowhere in the response = **gap confirmed** for
that property: `view-details` does not carry it, so `denodo_properties` must
also source from `property-management`.


In [9]:
import re


def strip_html_tags(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()


def get_view_details(view_name, db_name):
    """Return (status_code, json_or_None) instead of raising."""
    resp = auth_manager.get(
        BASE_URL + "/public/api/view-details",
        params={"viewName": view_name, "databaseName": db_name, "serverId": SERVER_ID},
    )
    if resp.ok:
        return resp.status_code, resp.json()
    return resp.status_code, None


_views_index = None  # lazy cache of the full views list, for diagnosing 404s

def lookup_view_in_list(view_name):
    global _views_index
    if _views_index is None:
        r = auth_manager.get(BASE_URL + "/public/api/views", params={"serverId": SERVER_ID})
        r.raise_for_status()
        _views_index = {v.get("name"): v for v in r.json()}
    return _views_index.get(view_name)


def extract_property_names(view_details):
    # BUG FIX (2026-07-27, found in Step 7): the item key is "propertyName",
    # NOT "name" -- in BOTH summaryPropertyMap and customTabPropertyMap.
    # The original prop.get("name") silently returned {None}; v10's
    # in_summaryPropertyMap column was only coincidentally correct. Any older
    # code reading summaryPropertyMap items via "name" inherits this bug.
    prop_groups = view_details.get("propertyInfo", {}).get("summaryPropertyMap", {})
    names = set()
    for group_name, props in prop_groups.items():
        for prop in props:
            names.add(prop.get("propertyName"))
    names.discard(None)
    return names


results = []
for group_id, sample in group_samples.items():
    expected = sample.get("expected_properties") or {}

    if not expected:
        results.append({
            "group": sample["group_name"], "view": sample["view"], "db": sample["db"],
            "status": "INCONCLUSIVE -- no populated Additional-tab values on this element",
            "gap_confirmed": None,
        })
        print(json.dumps(results[-1], indent=2)); print()
        continue

    status_code, details = get_view_details(sample["view"], sample["db"])
    if details is None:
        vinfo = lookup_view_in_list(sample["view"])
        if vinfo is None:
            diag = "view NOT in the views list at all -- stale property assignment?"
        else:
            diag = (f"view IS in the views list: db={vinfo.get('db')!r}, "
                    f"deleted={vinfo.get('deleted')!r}")
        results.append({
            "group": sample["group_name"], "view": sample["view"], "db": sample["db"],
            "status": f"view-details returned {status_code}; {diag}",
            "gap_confirmed": None,
        })
        print(json.dumps(results[-1], indent=2)); print()
        continue

    details_text = json.dumps(details)
    summary_names = extract_property_names(details)

    per_property = []
    for pname, pval in expected.items():
        val_stripped = strip_html_tags(pval)
        value_found = bool(
            (pval and pval in details_text)
            or (val_stripped and val_stripped in details_text)
        )
        per_property.append({
            "property": pname,
            "value_preview": (val_stripped or str(pval))[:80],
            "in_summaryPropertyMap": pname in summary_names,
            "name_anywhere_in_response": pname in details_text,
            "value_anywhere_in_response": value_found,
        })

    missing_values = [p["property"] for p in per_property
                      if not p["value_anywhere_in_response"]]

    results.append({
        "group": sample["group_name"], "view": sample["view"], "db": sample["db"],
        "per_property": per_property,
        "properties_with_value_missing_from_view_details": missing_values,
        "gap_confirmed": len(missing_values) > 0,
    })
    print(json.dumps(results[-1], indent=2)); print()


{
  "group": "Additional Information",
  "view": "admin_option_type_fvts",
  "db": "dataportal",
  "per_property": [
    {
      "property": "More Help for Web Services",
      "value_preview": "https://collaborate.lanl.gov/x/tYV4Cw",
      "in_summaryPropertyMap": false,
      "name_anywhere_in_response": true,
      "value_anywhere_in_response": true
    }
  ],
  "properties_with_value_missing_from_view_details": [],
  "gap_confirmed": false
}

{
  "group": "Data Card",
  "view": "bv_oc_footprints_alldesc_esr",
  "db": "dataportal",
  "status": "INCONCLUSIVE -- no populated Additional-tab values on this element",
  "gap_confirmed": null
}



In [10]:
conclusive = [r for r in results if r.get("gap_confirmed") is not None]
inconclusive = [r for r in results if r.get("gap_confirmed") is None]
any_gap = any(r["gap_confirmed"] for r in conclusive)

for r in inconclusive:
    print(f"INCONCLUSIVE  {r['group']} / {r['db']}.{r['view']}: {r['status']}")
if inconclusive:
    print()

if not results:
    print(
        "No non-Summary groups with assigned views were found. Either every "
        "group in this catalog is Summary-tab (question closes itself -- "
        "record 'N/A, checked empirically' in the contract), or the earlier "
        "TODOs need adjusting."
    )
elif not conclusive:
    print(
        "All comparisons were INCONCLUSIVE -- no view had non-null expected "
        "property values that could be checked against view-details. On this "
        "server the non-Summary groups exist but are effectively unpopulated; "
        "the gap question stays open until tested on an environment with "
        "populated Additional-tab properties (e.g. PROD once aliases settle)."
    )
elif any_gap:
    print(
        "GAP CONFIRMED: view-details omits at least one Additional-Properties-Tab "
        "property. denodo_properties (Step 1) must source from property-management "
        "in addition to view-details -- update the contract's source_field notes."
    )
else:
    print(
        "No gap found in the sample tested -- view-details appears to include "
        "non-Summary properties too. Worth widening the sample (more groups/views) "
        "before fully closing this question out in the contract."
    )


INCONCLUSIVE  Data Card / dataportal.bv_oc_footprints_alldesc_esr: INCONCLUSIVE -- no populated Additional-tab values on this element

No gap found in the sample tested -- view-details appears to include non-Summary properties too. Worth widening the sample (more groups/views) before fully closing this question out in the contract.


## Step 6 -- Locate WHERE in `view-details` the Additional-tab value lives

The v10 run showed the populated value (`More Help for Web Services` on
`admin_option_type_fvts`) is present in the response but **not** in
`summaryPropertyMap`. This cell finds the exact JSON path.

**Observed result (2026-07-27 run):**

```
Paths containing the VALUE (collaborate.lanl.gov):
  $.propertyInfo.customTabPropertyMap.Additional Information[0].visualValue
  $.propertyInfo.customTabPropertyMap.Additional Information[0].visualValueToEdit

Paths containing the property NAME (More Help for Web Services):
  $.propertyInfo.customTabPropertyMap.Additional Information[0].propertyName

propertyInfo keys: ['summaryPropertyMap', 'generalTabPropertyMap', 'customTabPropertyMap']
```

`propertyInfo` holds **three parallel maps** aligning with the `placeToShow` enum:
`SUMMARY_TAB` -> `summaryPropertyMap`, `SPECIFIC_CUSTOM_PROPERTY_TAB` ->
`customTabPropertyMap` (keyed by group name), plus `generalTabPropertyMap`.


In [ ]:
def find_paths(obj, needle, path="$"):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            hits += find_paths(v, needle, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits += find_paths(v, needle, f"{path}[{i}]")
    elif isinstance(obj, str) and needle in obj:
        hits.append(path)
    return hits


status_code, details = get_view_details("admin_option_type_fvts", "dataportal")

print("Paths containing the VALUE (collaborate.lanl.gov):")
for p in find_paths(details, "collaborate.lanl.gov"):
    print(" ", p)

print("\nPaths containing the property NAME (More Help for Web Services):")
for p in find_paths(details, "More Help for Web Services"):
    print(" ", p)

print("\npropertyInfo keys:", list(details.get("propertyInfo", {}).keys()))


## Step 7 -- Inspect the item schema of each `propertyInfo` map

**Observed result (2026-07-27 run):** `summaryPropertyMap` and
`customTabPropertyMap` share the **identical 16-key item schema**:

```
groupDescription, groupId, groupName, placeToShowGroup,
propertyDefaultValue, propertyDescription, propertyDescriptionType,
propertyId, propertyInterpolable, propertyName, propertyPossibleValues,
propertySearchDescription, propertyType, searchValue, visualValue,
visualValueToEdit
```

`generalTabPropertyMap` was empty on this view (schema exists, contents
unobserved). Because every item carries `placeToShowGroup`, `groupId`, and
`groupName` inline, a generic extractor over the union of the three maps needs
NO separate group-details calls.


In [ ]:
status_code, details = get_view_details("admin_option_type_fvts", "dataportal")
pinfo = details["propertyInfo"]

for map_name in ["summaryPropertyMap", "generalTabPropertyMap", "customTabPropertyMap"]:
    m = pinfo.get(map_name)
    print(f"=== {map_name} ===")
    if not m:
        print("  (empty)")
        continue
    for group_name, items in m.items():
        print(f"  group {group_name!r}: {len(items)} item(s)")
        if items:
            print(f"    item keys: {sorted(items[0].keys())}")
    print()


---
# Conclusions (2026-07-27)

**Question:** does `view-details` return property values for groups configured to
the *Additional Properties Tab*, or are they missing?

**Answer: they ARE returned -- but in `propertyInfo.customTabPropertyMap`, not
`summaryPropertyMap`.** The gap is at the *extraction* level, not the endpoint
level.

## Findings

1. Property groups carry a `placeToShow` field: `SUMMARY_TAB` (7 of 9 groups on
   this server) or `SPECIFIC_CUSTOM_PROPERTY_TAB` (2: **Additional Information**,
   **Data Card**).
2. `view-details` -> `propertyInfo` contains three parallel maps:
   `summaryPropertyMap`, `generalTabPropertyMap`, `customTabPropertyMap` --
   the last keyed by group name and carrying Additional-tab values (verified via
   full-response value search on `More Help for Web Services` /
   `admin_option_type_fvts`, elementId 5755).
3. Both observed maps share an identical 16-key item schema; the property name
   key is **`propertyName`** (NOT `name` -- a live bug in older extraction code,
   fixed in Step 5 of this version). `placeToShowGroup`/`groupId`/`groupName`
   are inline per item.
4. The collection endpoint `/groups/{id}/elements/{elementType}` requires
   `limit` + `offset` (ints) and `elementType="VIEWS"`; it identifies assigned
   elements but returns null property fields. The per-element endpoint returns
   `propertyId`/`visualValue` only; join names via `/groups/{id}/properties`.

## Implication for `denodo_properties` (Work item B)

`view-details` remains the single complete source for per-view property values.
The extractor must read the **union of all three maps** and use `propertyName`.

## Caveats

- All findings are from the temporary **`datacatalog-b`** server (9 property
  groups; PROD has 43 incl. 35 DCAT groups -- their absence on `-b` is
  unexplained and worth raising with Maxen).
- **Data Card stayed untestable** (no populated values on any view here); its
  sample view `bv_oc_footprints_alldesc_esr` also 404s on `view-details`
  (suspected soft-delete, unconfirmed).
- `generalTabPropertyMap` contents unobserved.
- **One PROD rerun is required to fully close this** -- see the v12 notebook.


## Remaining TODOs before treating this as conclusive

1. ~~Confirm `DISPLAY_LOCATION_FIELD` / `SUMMARY_VALUE`~~ **DONE (2026-07-27):**
   field is `placeToShow`, Summary value is `"SUMMARY_TAB"`.
2. ~~Confirm `ELEMENT_TYPE`~~ **DONE (2026-07-27):** Swagger confirms `"VIEWS"`;
   the 400s were caused by the missing **required** `limit` + `offset` query params.
3. ~~Confirm the response shape~~ **DONE (2026-07-27):** flat array of
   *(element, property)* rows (per Swagger's example value); Step 4 now regroups
   rows per view. `elementId` is included in the response.
4. If zero non-Summary groups exist in this catalog, this notebook's output
   *is* the answer -- no further probing needed, and no need to raise this
   with Maxen.

5. **Open after the v8 run (2026-07-27):** the collection endpoint returned
   `propertyName: null` on every row, and Data Card's sample view
   (`bv_oc_footprints_alldesc_esr`) 404s on `view-details` (likely soft-deleted).
   Step 4's raw dump, Step 4b's per-element cross-check, and Step 5's 404
   diagnosis in this version exist to resolve both.

6. **v10 (2026-07-27):** the per-element schema on this server is
   `propertyId`/`visualValue` only; Step 4b now joins against
   `/groups/{groupId}/properties` for names. One genuinely populated
   Additional-tab property exists (group 81, propertyId 159 on
   `admin_option_type_fvts`) -- the gap test runs on it via full-response
   value search in Step 5. Data Card remains untestable here (no populated
   values); note that for the PROD rerun.
